# 01 - Localització i Detecció de Caràcters
Aquest notebook implementa un pipeline clàssic per trobar matrícules basat en les característiques geomètriques de les lletres:

1. Alt contrast a la zona de la placa.
2. Caràcters de mida similar agrupats de prop.
3. Orientació horitzontal.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def mostrar_imatge(titol, imatge, cmap=None):
    plt.figure(figsize=(10, 4))
    plt.title(titol)
    if cmap:
        plt.imshow(imatge, cmap=cmap)
    else:
        plt.imshow(cv2.cvtColor(imatge, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

# Carregar la imatge original
img_original = cv2.imread('../data/raw/test_004.jpg')

mostrar_imatge("Input: Imatge amb il·luminació irregular", img_original)

## Pas 1: Adaptive Threshold (Alt contrast)
En lloc de fer un threshold global (que fallaria per culpa de les ombres), utilitzem **Adaptive Threshold**. Aquest mètode calcula el llindar òptim per a petites regions (finestres) de la imatge, garantint que extreu el text tant a les zones fosques com a les il·luminades.

In [ ]:
# Convertim a escala de grisos
gray = cv2.cvtColor(img_original, cv2.COLOR_BGR2GRAY)

# Configuració dels paràmetres (simulant el self.config)
config = {
    "thresh_window": 31, # Mida del bloc (ha de ser imparell)
    "thresh_offset": 15  # Constant restada a la mitjana calculada
}

# DO ADAPTIVE THRESHOLD (Codi de la imatge)
thresh = cv2.adaptiveThreshold(
    gray, 
    255, 
    cv2.ADAPTIVE_THRESH_MEAN_C, 
    cv2.THRESH_BINARY_INV, 
    config["thresh_window"], 
    config["thresh_offset"]
)

mostrar_imatge("1. Adaptive Threshold (Binary Inverted)", thresh, cmap='gray')

## Pas 2 i 3: Detect Blobs & Filter
Busquem els contorns continus (blobs). Com que l'Adaptive Threshold haurà generat molt de soroll al fons, filtrarem els contorns basant-nos en "Height and Weight" (Alçada i Amplada), i afegirem la relació d'aspecte.

In [ ]:
# DETECT ALL BLOBS (Codi de la imatge)
cnts, _ = cv2.findContours(thresh.copy(), cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

imatge_no_filtrada = img_original.copy()
for c in cnts:
    x, y, w, h = cv2.boundingRect(c)
    cv2.rectangle(imatge_no_filtrada, (x, y), (x+w, y+h), (0, 255, 0), 2)

mostrar_imatge("2. Detecció de Blobs (Sense Filtrar)", imatge_no_filtrada)


img_contours = img_original.copy()
blobs_filtrats = []

# FILTER BLOBS BASED ON HEIGHT AND WIDTH
alçada_img = img_original.shape[0]

for c in cnts:
    x, y, w, h = cv2.boundingRect(c)
    aspect_ratio = w / float(h)
    
    # Filtrem la brutícia:
    # 1. Alçada: ni molt petites (soroll) ni gegants (reixes)
    # 2. Aspect Ratio: Una lletra normalment és més alta que ampla (0.2 a 0.9)
    if (alçada_img * 0.02 < h < alçada_img * 0.8) and (0.2 < aspect_ratio < 1.5):
        blobs_filtrats.append((x, y, w, h))
        cv2.rectangle(img_contours, (x, y), (x+w, y+h), (255, 0, 0), 2)

mostrar_imatge("2 & 3. Detecció i Filtrat (Lletres candidates)", img_contours)
print(f"S'han filtrat i conservat {len(blobs_filtrats)} blobs candidats.")

## Pas 4: Group Filtered Blobs

Ara que tenim els candidats nets, busquem "Similar sized closely grouped characters" i "Horizontally oriented". Farem un agrupatge espacial cercant blobs que estiguin alineats en l'eix Y i a una distància constant en l'eix X.

In [ ]:
# Ordenem d'esquerra a dreta
blobs_filtrats.sort(key=lambda b: b[0])

# GROUP FILTER BLOBS
grups_horitzontals = []

for i in range(len(blobs_filtrats)):
    x1, y1, w1, h1 = blobs_filtrats[i]
    grup_actual = [(x1, y1, w1, h1)]
    
    for j in range(i + 1, len(blobs_filtrats)):
        x2, y2, w2, h2 = blobs_filtrats[j]
        
        # Validacions d'agrupació:
        # a) "Horizontally oriented": La Y del centre ha de ser gairebé igual
        dif_y = abs((y1 + h1/2) - (y2 + h2/2))
        
        # b) "Similar sized": Les altures (h) han de coincidir molt
        dif_h = abs(h1 - h2)
        
        # c) "Closely grouped": La distància (kerning) no pot ser enorme
        dist_x = x2 - (x1 + w1)
        
        # Si compleix els 3 criteris, formen part del mateix grup de text
        if dif_y < h1 * 0.3 and dif_h < h1 * 0.2 and 0 <= dist_x < h1 * 1.5:
            grup_actual.append((x2, y2, w2, h2))
            x1, y1, w1, h1 = x2, y2, w2, h2 # Actualitzem pel següent salt
            
    if len(grup_actual) >= 4: # Una matrícula sol tenir més de 4 lletres agrupades
        grups_horitzontals.append(grup_actual)

# Dibuixem el resultat final (El ROI de la matrícula)
img_final = img_original.copy()

if grups_horitzontals:
    # Ens quedem amb el grup que té més lletres
    millor_grup = max(grups_horitzontals, key=len)
    
    # Extraiem els límits totals del grup (Bounding Box de la Matrícula)
    xmin = min([b[0] for b in millor_grup])
    ymin = min([b[1] for b in millor_grup])
    xmax = max([b[0]+b[2] for b in millor_grup])
    ymax = max([b[1]+b[3] for b in millor_grup])
    
    # Dibuixem la caixa verda
    marge = 10
    cv2.rectangle(img_final, (xmin-marge, ymin-marge), (xmax+marge, ymax+marge), (0, 255, 0), 4)

mostrar_imatge("4. Grouping: Detecció final del grup horitzontal", img_final)

## Pas 5: Plate de-skew and rotate

Per redreçar la matrícula, descartarem el De-skew per transformació de perspectiva (ja que els extrems de la placa sovint són invisibles). En comptes d'això, utilitzarem la Rotació (Affine Transform), ja que dóna resultats més consistents.

Farem servir el centre del primer i l'últim caràcter del nostre grup validat per calcular l'angle exacte de la placa.

In [ ]:
# 1. Retallem la regió de la matrícula de la imatge original
# Utilitzem les coordenades xmin, ymin, xmax, ymax calculades al pas anterior
plate_region = img_original[max(0, ymin-marge) : min(alçada_img, ymax+marge), 
                            max(0, xmin-marge) : min(img_original.shape[1], xmax+marge)].copy()

mostrar_imatge("5a. Regió de la matrícula (abans de rotar)", plate_region)

# 2. Use characters to determine plate angle
# Agafem el primer i l'últim caràcter del grup
primer_char = millor_grup[0]
ultim_char = millor_grup[-1]

# Calculem els seus centres
cx1 = primer_char[0] + (primer_char[2] / 2.0)
cy1 = primer_char[1] + (primer_char[3] / 2.0)
cx2 = ultim_char[0] + (ultim_char[2] / 2.0)
cy2 = ultim_char[1] + (ultim_char[3] / 2.0)

# Calculem l'angle d'inclinació (dy / dx) en graus
dy = cy2 - cy1
dx = cx2 - cx1
degrees = np.degrees(np.arctan2(dy, dx))
print(f"Angle d'inclinació detectat: {degrees:.2f} graus")

# 3. Rotate
h, w = plate_region.shape[:2]
cX, cY = w // 2, h // 2

# Generem la matriu de rotació i apliquem la transformació afí
M = cv2.getRotationMatrix2D((cX, cY), degrees, 1.0)

# borderValue=(180, 180, 180) omple els buits girats amb un gris neutre per no afegir vores negres
rotated = cv2.warpAffine(plate_region, M, (w, h), borderValue=(180, 180, 180))

mostrar_imatge(f"5b. Matrícula redreçada (Rotada {degrees:.2f}°)", rotated)

### Guardem el resultat

Ara que ja tenim el resultat, anem a guardar-lo a una carpeta per utilitzar-lo més avant amb la **Notebook 2 Character Segmentation**.

In [ ]:
# Guardem la imatge final de la matrícula redreçada
PLATE_OUTPUT_PATH = 'data/plate_output.jpg'

cv2.imwrite(PLATE_OUTPUT_PATH, rotated)
print(f"Matrícula redreçada guardada a: {PLATE_OUTPUT_PATH}")